# NSE standalone financial data: coverage and project scope

Use this notebook to decide which companies, industries, dates, and predictors your data can support.
It profiles source data without changing or imputing the original financial values. It does not train models.

**Run in order:** configure dates, upload your yearly `company_quarterly_financials_*.csv` files,
review quality checks, and inspect scope scenarios. You may instead read a folder in Google Drive.
Empty yearly files are reported and skipped. File-year labels are never treated as reporting years.

An optional `industry_mapping.csv` can provide verified industry labels with columns
`NSE_SYMBOL, INDUSTRY, SECTOR, IS_FINANCIAL`. Only `NSE_SYMBOL` and `INDUSTRY` are required.
Supply `IS_FINANCIAL` as True/False (or 1/0) to explicitly identify financial companies.
Do not upload fabricated classifications. BANKING/NBFC filenames are indicators, not verified industry labels.

Input methods follow [Google Colab's official examples](https://colab.research.google.com/notebooks/io.ipynb).


In [ ]:
import io
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

INPUT_MODE = 'drive'  # 'upload' or 'drive'
DRIVE_FOLDER = '/content/drive/MyDrive/nse-financial-research/data/interim'
START_DATE = '2022-04-01'
END_DATE = '2024-12-31'
BASIS = 'Standalone'
TARGET_COMPANIES = 100
FINANCIAL_COMPANY_THRESHOLD = 500
MIN_USABLE_QUARTERS = 11  # Planning threshold, not a power calculation.
MIN_CONSECUTIVE_QUARTERS = 8
MIN_CORE_COMPLETENESS = 0.80
CORE_FIELDS = ['TOTAL_INCOME', 'PROFIT_BEFORE_TAX', 'TAX_EXPENSE']
OUTPUT_DIR = Path('/content/drive/MyDrive/nse-financial-research/outputs/quality')
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_rows', 40)

start = pd.Timestamp(START_DATE)
end = pd.Timestamp(END_DATE)
assert start <= end, 'START_DATE must not be after END_DATE.'
calendar_quarters = pd.period_range(start, end, freq='Q')


## 1. Load the source files

Upload multiple CSVs together, including the optional mapping file. Only financial files with the
expected filename pattern are loaded. Drive mode searches the selected folder and its subfolders.
Files with invalid schemas or parsing errors are reported rather than silently dropping bad rows.


In [ ]:
if INPUT_MODE == 'upload':
    from google.colab import files
    uploaded = files.upload()
    inputs = [(name, io.BytesIO(data)) for name, data in uploaded.items()]
elif INPUT_MODE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    inputs = [(str(p), p) for p in sorted(Path(DRIVE_FOLDER).rglob('*.csv'))
              if p.name.startswith('company_quarterly_financials_')
              or p.name == 'industry_mapping.csv']
else:
    raise ValueError("Choose INPUT_MODE = 'drive' or 'drive'.")


In [ ]:
loaded, file_checks, mapping_frames = [], [], []
required = {'NSE_SYMBOL', 'PERIOD_END', 'STANDALONE_CONSOLIDATED'}
for name, source in inputs:
    filename = Path(name).name
    if not (filename.startswith('company_quarterly_financials_') or filename == 'industry_mapping.csv'):
        continue
    try:
        frame = pd.read_csv(source, dtype=str, low_memory=False, encoding='utf-8-sig')
        frame.columns = frame.columns.str.strip().str.upper()
        if frame.columns.duplicated().any():
            raise ValueError('Duplicate column labels after normalization.')
        if filename == 'industry_mapping.csv':
            mapping_frames.append(frame)
            continue
        missing = required - set(frame.columns)
        if missing:
            raise ValueError('Required columns missing: ' + ', '.join(sorted(missing)))
        frame['_INPUT_FILE'] = name
        frame['_SOURCE_ROW'] = np.arange(2, len(frame) + 2)
        loaded.append(frame)
        status = 'Loaded' if len(frame) else 'Header only'
        file_checks.append({'file': name, 'status': status, 'rows': len(frame), 'detail': ''})
    except pd.errors.EmptyDataError:
        file_checks.append({'file': name, 'status': 'Empty', 'rows': 0, 'detail': 'No CSV columns'})
    except (ValueError, UnicodeError, pd.errors.ParserError) as exc:
        file_checks.append({'file': name, 'status': 'Failed', 'rows': 0, 'detail': str(exc)})
file_checks = pd.DataFrame(file_checks, columns=['file', 'status', 'rows', 'detail'])
display(file_checks)
if not loaded:
    raise ValueError('No valid financial files loaded. Check filenames and required columns.')
raw = pd.concat(loaded, ignore_index=True, sort=False)
data_columns = [c for c in raw.columns if not c.startswith('_')]
for col in ['SOURCE_FILE', 'PAT', 'AUDITED_FLAG', 'PERIOD_START', 'ROUNDING',
            'RESULT_APPROVAL_DATE', 'INFORMATION_DATE_FOR_MODEL']:
    if col not in raw:
        raw[col] = pd.NA
raw['NSE_SYMBOL'] = raw.NSE_SYMBOL.str.strip().str.upper().replace('', pd.NA)
raw['_BASIS'] = raw.STANDALONE_CONSOLIDATED.str.strip().str.casefold()
raw['_PERIOD_END'] = pd.to_datetime(raw.PERIOD_END, errors='coerce')
raw['_PERIOD_START'] = pd.to_datetime(raw.PERIOD_START, errors='coerce')
raw['_FORMAT'] = raw.SOURCE_FILE.str.extract(r'^([^_]+)')[0].str.upper().fillna('UNKNOWN')
raw['_PAT'] = pd.to_numeric(raw.PAT, errors='coerce')
raw['_INVALID_PAT'] = raw.PAT.notna() & raw.PAT.str.strip().ne('') & raw._PAT.isna()
selected = raw.loc[raw._BASIS.eq(BASIS.casefold())].copy()
scoped = selected.loc[selected._PERIOD_END.between(start, end)].copy()
scoped['_Q'] = scoped._PERIOD_END.dt.to_period('Q')
print(f'All records: {len(raw):,}; selected basis: {len(selected):,}; in date window: {len(scoped):,}')
print(f'Company symbols in date window: {scoped.NSE_SYMBOL.nunique():,}')
if scoped.empty:
    raise ValueError('No records match BASIS and the date window. Revise settings or upload coverage.')


## 2. Reporting basis, dates, audit status, and filing formats

These are record counts, not unique companies or confirmed quarterly samples.
A filename prefix describes a filing format; INDAS must not be labeled non-financial.
Calendar quarter assignments below are provisional until each company's financial calendar and filing context are checked.


In [ ]:
basis_summary = raw.groupby('_BASIS', dropna=False).agg(
    records=('NSE_SYMBOL', 'size'), company_symbols=('NSE_SYMBOL', 'nunique')).reset_index()
period_summary = scoped.groupby(scoped._PERIOD_END.dt.year.rename('reporting_year')).agg(
    records=('NSE_SYMBOL', 'size'), company_symbols=('NSE_SYMBOL', 'nunique'),
    populated_pat=('_PAT', 'count')).reindex(range(start.year, end.year + 1), fill_value=0).reset_index()
audit_summary = scoped.groupby('AUDITED_FLAG', dropna=False).agg(
    records=('NSE_SYMBOL', 'size'), company_symbols=('NSE_SYMBOL', 'nunique')).reset_index()
format_summary = scoped.groupby('_FORMAT', dropna=False).agg(
    records=('NSE_SYMBOL', 'size'), company_symbols=('NSE_SYMBOL', 'nunique'),
    populated_pat=('_PAT', 'count')).reset_index()
format_summary['pat_completeness_pct'] = (100 * format_summary.populated_pat / format_summary.records).round(1)
for label, table in [('Reporting basis: all inputs', basis_summary),
                     ('Reporting years: selected basis and dates', period_summary),
                     ('Audit status: scoped records', audit_summary),
                     ('Filing formats: scoped records', format_summary)]:
    print(label)
    display(table)
ax = period_summary.plot.bar(x='reporting_year', y='company_symbols', legend=False,
                            color='#18766b', figsize=(8, 3))
ax.set(title='Company symbols with records by reporting year', xlabel='Reporting year', ylabel='Symbols')
plt.tight_layout()
plt.show()


## 3. Field completeness and quality blockers

Missing data are not zeros. No missing PAT is replaced with profit before tax minus tax.
That difference can be a reconciliation diagnostic, but the actual target must be verified against source facts.
Units may already have been scaled during extraction: do not multiply amounts again without checking the parser.


In [ ]:
planned_fields = ['PAT', 'TOTAL_INCOME', 'REVENUE', 'OPERATING_PROFIT', 'EPS_BASIC',
                  'ASSETS', 'LIABILITIES', 'EQUITY', 'DEBT', 'PROFIT_BEFORE_TAX', 'TAX_EXPENSE']
completeness_rows = []
for field in planned_fields:
    exists = field in scoped.columns
    values = pd.to_numeric(scoped[field], errors='coerce') if exists else pd.Series(np.nan, index=scoped.index)
    completeness_rows.append({'field': field, 'column_present': exists,
                              'numeric_records': int(values.notna().sum()),
                              'numeric_pct': round(100 * values.notna().mean(), 1)})
field_completeness = pd.DataFrame(completeness_rows)
display(field_completeness)
duplicates_exact = int(raw.duplicated(subset=data_columns).sum())
keys = ['NSE_SYMBOL', '_PERIOD_END', '_BASIS']
duplicate_mask = scoped.duplicated(keys, keep=False)
duplicate_records = scoped.loc[duplicate_mask].copy()
print('Exact repeats across inputs:', duplicates_exact)
print('Scoped records in repeated company/date/basis groups:', int(duplicate_mask.sum()))
print('Missing dates within selected basis:', int(selected._PERIOD_END.isna().sum()))
print('Missing symbols in scoped records:', int(scoped.NSE_SYMBOL.isna().sum()))
print('Nonnumeric populated PAT:', int(scoped._INVALID_PAT.sum()))
print('PAT zero / negative:', int(scoped._PAT.eq(0).sum()), '/', int(scoped._PAT.lt(0).sum()))
print('Units / rounding labels:')
display(scoped.ROUNDING.fillna('MISSING').value_counts().rename_axis('rounding').reset_index(name='records'))
scoped['_DURATION_DAYS'] = (scoped._PERIOD_END - scoped._PERIOD_START).dt.days + 1
if 'QUARTER' in scoped:
    duration_summary = scoped.groupby('QUARTER', dropna=False)._DURATION_DAYS.agg(['count', 'min', 'median', 'max']).reset_index()
    display(duration_summary)
else:
    duration_summary = pd.DataFrame()
timing_fields = ['RESULT_APPROVAL_DATE', 'INFORMATION_DATE_FOR_MODEL', 'PRIOR_INTIMATION_DATE']
display(pd.DataFrame([{'field': c, 'populated_records': int(scoped[c].notna().sum()) if c in scoped else 0}
                      for c in timing_fields]))
print('Approval and prior-intimation dates are not automatically public information-availability dates.')


## 4. Optional industry mapping

Verified labels must come from a documented company master or another reliable source.
If no mapping is uploaded, the notebook produces filing-format groups only.
Mapping rows must be unique by symbol; resolve renamed symbols and conflicting classifications before upload.
Symbols are a provisional identifier, not proof of distinct legal entities across six years.


In [ ]:
companies = scoped[['NSE_SYMBOL']].dropna().drop_duplicates().copy()
if mapping_frames:
    mapping = pd.concat(mapping_frames, ignore_index=True)
    if not {'NSE_SYMBOL', 'INDUSTRY'}.issubset(mapping.columns):
        raise ValueError('Mapping requires NSE_SYMBOL and INDUSTRY.')
    mapping['NSE_SYMBOL'] = mapping.NSE_SYMBOL.str.strip().str.upper()
    mapping = mapping.dropna(subset=['NSE_SYMBOL']).drop_duplicates()
    if mapping.NSE_SYMBOL.duplicated().any():
        raise ValueError('Mapping has multiple rows for a symbol. Resolve them before merging.')
    keep = [c for c in ['NSE_SYMBOL', 'INDUSTRY', 'SECTOR', 'IS_FINANCIAL'] if c in mapping]
    companies = companies.merge(mapping[keep], on='NSE_SYMBOL', how='left', validate='one_to_one')
if 'INDUSTRY' not in companies:
    companies['INDUSTRY'] = pd.NA
companies['INDUSTRY'] = companies.INDUSTRY.str.strip().replace({'': pd.NA, '-': pd.NA}).fillna('UNMAPPED')
if 'IS_FINANCIAL' in companies:
    flag = companies.IS_FINANCIAL.astype('string').str.strip().str.lower()
    companies['_VERIFIED_FINANCIAL'] = flag.map({'true': True, '1': True, 'yes': True,
                                                'false': False, '0': False, 'no': False}).astype('boolean')
else:
    companies['_VERIFIED_FINANCIAL'] = pd.Series(pd.NA, index=companies.index, dtype='boolean')
format_sets = scoped.dropna(subset=['NSE_SYMBOL']).groupby('NSE_SYMBOL')._FORMAT.agg(lambda x: '|'.join(sorted(set(x))))
companies = companies.merge(format_sets.rename('FILING_FORMATS'), on='NSE_SYMBOL', how='left')
companies['_FINANCIAL_FORMAT_INDICATOR'] = companies.FILING_FORMATS.str.split('|').apply(
    lambda values: bool(set(values) & {'BANKING', 'NBFC'}))
industry_counts = companies.groupby('INDUSTRY').agg(company_symbols=('NSE_SYMBOL', 'nunique')).reset_index()
industry_counts = industry_counts.sort_values('company_symbols', ascending=False)
display(industry_counts)
print('Financial-format candidates:', int(companies._FINANCIAL_FORMAT_INDICATOR.sum()))
print('Financial companies explicitly marked in supplied mapping:', int(companies._VERIFIED_FINANCIAL.fillna(False).sum()))
print('Unmapped company symbols:', int(companies.INDUSTRY.eq('UNMAPPED').sum()))


## 5. Consecutive-quarter and candidate target coverage

This is a conservative screening exercise, not a cleaned modeling dataset.
A company-quarter is provisionally usable only when it has one record, numeric PAT, enough core fields,
and a known start/end interval of 70-110 days. The duration range is a screening rule, not proof that facts are quarterly.
Multiple records per company-quarter are flagged for resolution, not arbitrarily kept or averaged.

Candidate adjacent PAT pairs require two immediately consecutive usable quarters and nonzero base PAT.
Counts do not establish point-in-time availability, unit consistency, target validity, or statistical independence.


In [ ]:
existing_core = [c for c in CORE_FIELDS if c in scoped]
numeric_core = pd.DataFrame({c: pd.to_numeric(scoped[c], errors='coerce') for c in existing_core}, index=scoped.index)
scoped['_CORE_FRACTION'] = numeric_core.notna().sum(axis=1) / max(len(CORE_FIELDS), 1)
work = scoped.dropna(subset=['NSE_SYMBOL']).copy()
work['_GROUP_SIZE'] = work.groupby(['NSE_SYMBOL', '_Q']).NSE_SYMBOL.transform('size')
work['_CANDIDATE_USABLE'] = (work._GROUP_SIZE.eq(1) & work._PAT.notna()
                           & work._CORE_FRACTION.ge(MIN_CORE_COMPLETENESS)
                           & work._DURATION_DAYS.between(70, 110))
def longest_run(ordinals):
    values = sorted(set(ordinals))
    best = current = 0
    previous = None
    for value in values:
        current = current + 1 if previous is not None and value == previous + 1 else 1
        best = max(best, current)
        previous = value
    return best
coverage_rows = []
for symbol, group in work.groupby('NSE_SYMBOL'):
    observed = set(group._Q.astype('int64'))
    usable = group.loc[group._CANDIDATE_USABLE].sort_values('_Q')
    usable_q = set(usable._Q.astype('int64'))
    nonzero_q = set(usable.loc[usable._PAT.ne(0), '_Q'].astype('int64'))
    pairs = sum(q + 1 in usable_q for q in nonzero_q)
    coverage_rows.append({'NSE_SYMBOL': symbol, 'records': len(group),
                          'observed_quarters': len(observed),
                          'observed_coverage_pct': round(100 * len(observed) / len(calendar_quarters), 1),
                          'quarters_with_numeric_pat': group.loc[group._PAT.notna(), '_Q'].nunique(),
                          'candidate_usable_quarters': len(usable_q),
                          'longest_usable_run': longest_run(usable_q),
                          'candidate_adjacent_pat_pairs': pairs,
                          'duplicate_quarters': int(group.groupby('_Q').size().gt(1).sum())})
coverage = pd.DataFrame(coverage_rows).merge(companies, on='NSE_SYMBOL', validate='one_to_one')
coverage['CANDIDATE_ELIGIBLE'] = (coverage.candidate_usable_quarters.ge(MIN_USABLE_QUARTERS)
                                & coverage.longest_usable_run.ge(MIN_CONSECUTIVE_QUARTERS)
                                & coverage.candidate_adjacent_pat_pairs.gt(0))
display(coverage.sort_values(['CANDIDATE_ELIGIBLE', 'candidate_usable_quarters'], ascending=False).head(30))
print('Missing core columns:', sorted(set(CORE_FIELDS) - set(scoped.columns)))
print('Provisional candidate companies:', int(coverage.CANDIDATE_ELIGIBLE.sum()))
print('A low count may reflect extraction errors, not absence of economic data.')


## 6. Compare scope options

The 500-company threshold is your planning preference, not a statistical requirement.
A financial-format screen cannot certify a financial-sector sample. Industry-specific scopes need verified mappings.
Every option also requires a later-period test set, sufficient targets, and checks for reporting comparability.


In [ ]:
scope_rows = []
scopes = [('All selected-basis companies', pd.Series(True, index=coverage.index), TARGET_COMPANIES),
          ('BANKING/NBFC format candidates (unverified sector)', coverage._FINANCIAL_FORMAT_INDICATOR, FINANCIAL_COMPANY_THRESHOLD),
          ('Financial companies explicitly mapped', coverage._VERIFIED_FINANCIAL.fillna(False), FINANCIAL_COMPANY_THRESHOLD)]
for industry in sorted(coverage.loc[coverage.INDUSTRY.ne('UNMAPPED'), 'INDUSTRY'].unique()):
    scopes.append(('Industry: ' + industry, coverage.INDUSTRY.eq(industry), TARGET_COMPANIES))
for name, mask, requested in scopes:
    subset = coverage.loc[mask]
    eligible = subset.loc[subset.CANDIDATE_ELIGIBLE]
    scope_rows.append({'scope': name, 'observed_symbols': len(subset),
                       'candidate_eligible_symbols': len(eligible), 'planning_target': requested,
                       'target_met_provisionally': len(eligible) >= requested,
                       'candidate_pat_pairs': int(eligible.candidate_adjacent_pat_pairs.sum())})
scope_summary = pd.DataFrame(scope_rows)
display(scope_summary)
missing_years = period_summary.loc[period_summary.records.eq(0), 'reporting_year'].tolist()
print('Years without selected-basis records:', missing_years)
print('Before finalizing scope: verify PAT extraction; resolve duplicate filings; validate units and quarterly facts;')
print('establish actual public availability dates; choose a stable company identifier; then screen train/test coverage.')
print('No automatic recommendation is made solely from company or record counts.')


## 7. Export the analysis

The ZIP contains file checks, completeness, industry/format summaries, company coverage, scope scenarios,
and duplicate records with source references. It does not alter input files or certify a modeling-ready dataset.
Choose a scope only after resolving the blockers; do not promise six-year coverage from filenames alone.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
exports = {'file_checks.csv': file_checks, 'basis_summary.csv': basis_summary,
           'period_summary.csv': period_summary, 'audit_summary.csv': audit_summary,
           'format_summary.csv': format_summary, 'field_completeness.csv': field_completeness,
           'industry_counts.csv': industry_counts, 'company_coverage.csv': coverage,
           'scope_scenarios.csv': scope_summary, 'duplicate_records.csv': duplicate_records,
           'duration_summary.csv': duration_summary}
for filename, table in exports.items():
    table.to_csv(OUTPUT_DIR / filename, index=False)
report = [f'# Project scope screening: {BASIS}',
          f'Dates: {START_DATE} to {END_DATE}',
          f'Input rows: {len(raw):,}; scoped rows: {len(scoped):,}',
          f'Scoped symbols: {scoped.NSE_SYMBOL.nunique():,}',
          f'Candidate eligibility: at least {MIN_USABLE_QUARTERS} usable quarters and {MIN_CONSECUTIVE_QUARTERS} consecutive quarters.',
          'These are planning rules, not a power calculation or final eligibility certification.',
          f'Core fields: {CORE_FIELDS}; completeness threshold: {MIN_CORE_COMPLETENESS:.0%}.',
          'Duplicate quarters are conservatively excluded until resolved; known 70-110 day durations are required.',
          'Financial-format indicators do not replace verified sector classifications.',
          'PAT units, source mappings, availability dates, and reporting contexts remain to be verified.',
          'Symbol counts may differ from legal-entity counts because of renaming and corporate changes.',
          'Review the exported company coverage and scope scenario tables before choosing the population.']
(OUTPUT_DIR / 'scope_review_notes.md').write_text('\n\n'.join(report), encoding='utf-8')
archive_path = OUTPUT_DIR.parent / 'scope_analysis_results.zip'
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for filename in list(exports) + ['scope_review_notes.md']:
        archive.write(OUTPUT_DIR / filename, arcname=filename)
print('Results saved:', archive_path)
if INPUT_MODE in ['upload', 'drive']:
    from google.colab import files
    files.download(str(archive_path))
